# Question

In the Premier League, how strongly asosciated is a team's annual wages associated with their performance in a season? Additionally, do clubs beat or miss what their wages predicted?

INTEREST: 

A club with a lot to spend might still be outperformed by smarter teams in recruitment, coaching, and overall well-being of players. I think back to Leicester City's run back in 2015-2016, where the underdogs had one of the smaller annual wages, but the coach's tactics and team's performance proved otherwise. 

WAGES:

I collected the data from FBref on October 16, 2026. Annual wages are defined in pounds and are estimated for earlier years because of the unkown circumstances of the clubs' wages. The scope is 10 seasons (2013-2024), where there are 200 club-seasons. the numbers are not an esitmate of first-team pay and are not audited club payroll 

# PREDICTIONS:

I expect teams like Manchecter City, Manchester United, Chelsea, Liverpool, Tottenham, and Arsenal to overperform their wages based on the clubs' reputations and teams that are predicted to perform poorly based on performance stats will most likely underperform their wages. However, I expect the relationship to be relatively strong because the more time owners put into making payment to keep players around should improve the team's overall performance. There are some confounding variables to be aware of, like the relationship between owner or club, or the players having beef between each other. 

In [62]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px


from season_loader import load_season
from validators import validate_season

In [59]:
df = pd.read_csv(Path('data/processed/merged_datasets/merged_wages_points.csv'))

In [60]:
print(df.head())
print(df.shape)
print(df.dtypes)
print(df.columns)


   Wage_Rk            Squad  # Pl  % Estimated   Season  Points_Rk  MP   W  D  \
0        1  Manchester City    26          100  2013_14          1  38  27  5   
1        2          Chelsea    30          100  2013_14          3  38  25  7   
2        3   Manchester Utd    31          100  2013_14          7  38  19  7   
3        4          Arsenal    32          100  2013_14          4  38  24  7   
4        5        Liverpool    26          100  2013_14          2  38  26  6   

    L  GD  Pts  Wages_GBP  Wages_EUR  Wages_USD  
0   6  65   86  113254000  135062723  137628660  
1   6  44   82  102922000  122741146  125072997  
2  12  21   64   97292000  116027007  118231300  
3   7  27   79   71994000   85857503   87488634  
4   6  51   84   65647000   78288296   79775627  
(200, 15)
Wage_Rk         int64
Squad          object
# Pl            int64
% Estimated     int64
Season         object
Points_Rk       int64
MP              int64
W               int64
D               int64
L    

In [61]:
print(df['Season'].nunique())    # Checks the number of unique seasons in the dataset
print(df['Season'].value_counts()) # Checks the count of each unique season in the dataset
print(df['Wages_GBP'].describe())  # Provides descriptive statistics for the 'Wages_GBP' column

10
Season
2013_14    20
2014_15    20
2015_16    20
2016_17    20
2017_18    20
2018_19    20
2019_20    20
2020_21    20
2021_22    20
2022_23    20
Name: count, dtype: int64
count    2.000000e+02
mean     7.252417e+07
std      4.382432e+07
min      1.558100e+07
25%      4.127500e+07
50%      5.593750e+07
75%      9.474700e+07
max      2.388500e+08
Name: Wages_GBP, dtype: float64


# DERIVED VARIABLES
Raw wages aren't comparable across years because of inflation over the years, and the distribution would be heavily skewed_right. 

In [55]:
df['Wage_Share'] = df.groupby('Season')['Wages_GBP'].transform(lambda x: x / x.sum())
print(df.groupby('Season')['Wage_Share'].sum())  # This should print 1.0 for each season if the calculation is correct
# Wage share helps handle inflation between seasons by normalizing wages within each season. It allows for a fair comparison of wages across different seasons, as it accounts for the total wage pool in each season rather than absolute values.

Season
2013_14    1.0
2014_15    1.0
2015_16    1.0
2016_17    1.0
2017_18    1.0
2018_19    1.0
2019_20    1.0
2020_21    1.0
2021_22    1.0
2022_23    1.0
Name: Wage_Share, dtype: float64


In [56]:
df['Log_Wage_Share'] = df['Wage_Share'].apply(lambda x: np.log(x) if x > 0 else np.nan)  # Apply log transformation to 'Wage_Share', handling non-positive values
print(df['Log_Wage_Share'].describe())  # Provides descriptive statistics for the 'Log_Wage_Share' column

count    200.000000
mean      -3.140276
std        0.537023
min       -4.418562
25%       -3.512740
50%       -3.176418
75%       -2.728854
max       -1.964533
Name: Log_Wage_Share, dtype: float64


In [57]:
df['Wages_Rank'] = df.groupby('Season')['Wages_GBP'].rank(method='min')

# Exploratory Data Analysis

## Distributions:

In [8]:
# Histogram of Raw Wages
fig1 = px.histogram(df, x='Wages_GBP', title='Distribution of Raw Wages', labels={'Wages_GBP': 'Wages (GBP)', 'y': 'Number of Teams'})
fig1.show()


Observations: The distribution is skewed right with more teams' annual wages ranging around the 50 million pounds. There are two teams as outliers that go beyond 200 million pounds, but the average annual wages is greater than the median of the wages given amongst clubs.  

In [64]:
# Histogram of Log Wage Share
df['Wage_Share'] = df.groupby('Season')['Wages_GBP'].transform(lambda x: x / x.sum())
df['Log_Wage_Share'] = df['Wage_Share'].apply(lambda x: np.log(x) if x > 0 else np.nan)
fig2 = px.histogram(df, x ='Log_Wage_Share', title='Distribution of Log Wage Share', labels={'Log_Wage_Share': 'Log Wage Share', 'y': 'Number of Teams'})
fig2.show()

Observations: Logging the values of the Wage Shares creates a more normal distribution, making the ratios more equidistant. 

In [66]:
# Histogram of Points
fig3 = px.histogram(df, x='Pts', title='Distribution of Points Earned', labels={'Pts': 'Points', 'y': 'Number of Teams'})
fig3.show()

Observations: The points are little bit skewed to the right where most teams appear to be performing below standards compared to their wages. The median number of points would be around 40-50 points which is a little surprising because a lot of owners invest a lot at high levels to make sure teams have access to perform stronger. 

## League-wide wages over time

In [67]:
# Line Plot of Average Wages Per Season
avg_wages_per_season = df.groupby('Season')['Wages_GBP'].mean().reset_index()
fig4 = px.line(avg_wages_per_season, x='Season', y='Wages_GBP', title='Average Wages Per Season', labels={'Wages_GBP': 'Average Wages (GBP)', 'Season': 'Season'})
fig4.show()

Observations: The massive jump of less than 50 million pound average wages compared to the above-90 million pound averages suggests the need to create Wage Share data to account for inflation. 

## Scatter Bubble Plot

In [82]:
# Scatterplot of Raw Wages vs Points (Clubs are hovered over to show club names)
fig5 = px.scatter(df, x='Wages_GBP', y='Pts', hover_data=['Squad', 'Season'], color = 'Season', title='Raw Wages vs Points', labels={'Wages_GBP': 'Wages (GBP)', 'Pts': 'Points'})
fig5.show()

**Observations:** There seems to be a correlation between teams spending more on wages anually performing better, where teams in-between 100 to 150 million earn over 80 points which is enough for Champions League Football. However, some teams that break the budget end up underperforming, causing pressure of big spending equaling poor results. 

In [83]:
# Scatterplot of Wage Share vs Points (Clubs are hovered over to show club names)
fig6 = px.scatter(df, x='Wage_Share', y='Pts', hover_data=['Squad', 'Season'], color = 'Season', title='Wage Shares vs Points', labels={'Share of league wage bill': 'Share of league wage bill', 'Pts': 'Points'})
fig6.show()

**Observations:** The x-axis shows that club’s share of the league’s total wage bill in that season, written as a fraction instead of a percentage. Most dots sit between about 0.02 and 0.06. Only a few stretch out toward 0.10 to 0.14, with the farthest at about 14% (probably Manchester United in 2021-22, given the 58 points, but hover to confirm). That long right tail is the skew the log transform is meant to address. Because this axis is linear, the crowded left side is hard to read and a few rich clubs stretch the right side out.

In [84]:
# Scatterplot of Log Wage Share vs Points (Clubs are hovered over to show club names)
fig7 = px.scatter(df, x='Log_Wage_Share', y='Pts', hover_data=['Squad', 'Season'], color = 'Season', title='Log Wage Shares vs Points', labels={'Log_Wage_Share': 'Log Wage Share', 'Pts': 'Points'})
fig7.show()

**Oberservations:** As the log accounts for inflation, low spenders are limited in how many points they get, while big spenders can finish anywhere from title winners to underachievers. For Chelsea (2022-2023), they spent high compared to other teams' annual wages, but the big spending caused them to fall short from even reaching European competitions.